<a href="https://colab.research.google.com/github/kempegowda7/BIS-lab1/blob/main/bis_bm24cs136_ACO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np

def warehouse_aco_picking():
    print("--- Warehouse ACO Configuration ---")
    num_shelves = int(input("Enter the number of target shelves to visit: "))

    print("\nEnter Packing Station coordinates:")
    px = float(input("  X coordinate: "))
    py = float(input("  Y coordinate: "))
    coords = [[px, py]]

    for i in range(num_shelves):
        print(f"\nEnter coordinates for Shelf {i+1}:")
        sx = float(input("  X coordinate: "))
        sy = float(input("  Y coordinate: "))
        coords.append([sx, sy])

    shelf_coordinates = np.array(coords)
    num_nodes = len(shelf_coordinates)

    dist_matrix = np.zeros((num_nodes, num_nodes))
    for i in range(num_nodes):
        for j in range(num_nodes):
            dist_matrix[i, j] = np.sum(np.abs(shelf_coordinates[i] - shelf_coordinates[j]))

    num_ants = 5
    alpha = 1.0
    beta = 2.0
    rho = 0.4
    Q = 100
    max_iterations = 50

    pheromones = np.ones((num_nodes, num_nodes)) * 0.1
    heuristic = 1.0 / (dist_matrix + 1e-9)

    global_best_distance = float('inf')
    global_best_route = []

    for iteration in range(max_iterations):
        iteration_routes = []
        iteration_distances = []

        for ant in range(num_ants):
            current_node = 0
            unvisited_shelves = list(range(1, num_nodes))
            route = [current_node]

            while unvisited_shelves:
                probabilities = []
                for target in unvisited_shelves:
                    tau = pheromones[current_node, target] ** alpha
                    eta = heuristic[current_node, target] ** beta
                    probabilities.append(tau * eta)

                sum_prob = sum(probabilities)
                probabilities = [p / sum_prob for p in probabilities]

                next_node = np.random.choice(unvisited_shelves, p=probabilities)
                route.append(next_node)
                unvisited_shelves.remove(next_node)
                current_node = next_node

            route.append(0)
            iteration_routes.append(route)

            total_dist = sum(dist_matrix[route[i], route[i+1]] for i in range(len(route) - 1))
            iteration_distances.append(total_dist)

            if total_dist < global_best_distance:
                global_best_distance = total_dist
                global_best_route = route

        pheromones *= (1 - rho)
        for route, dist in zip(iteration_routes, iteration_distances):
            pheromone_deposit = Q / dist
            for i in range(len(route) - 1):
                pheromones[route[i], route[i+1]] += pheromone_deposit
                pheromones[route[i+1], route[i]] += pheromone_deposit

    print("\n" + "="*40)
    print("      ACO PICKING OPTIMIZATION RESULTS     ")
    print("="*40)
    print(f"Total Iterations Run   : {max_iterations}")
    print(f"Optimal Visiting Route : {' -> '.join(map(str, global_best_route))}")
    print(f"Minimum Tour Distance  : {global_best_distance:.2f} meters")
    print("="*40)

if __name__ == "__main__":
    warehouse_aco_picking()


--- Warehouse ACO Configuration ---
Enter the number of target shelves to visit: 3

Enter Packing Station coordinates:
  X coordinate: 0
  Y coordinate: 0

Enter coordinates for Shelf 1:
  X coordinate: 2
  Y coordinate: 5

Enter coordinates for Shelf 2:
  X coordinate: 8
  Y coordinate: 1

Enter coordinates for Shelf 3:
  X coordinate: 4
  Y coordinate: 9

      ACO PICKING OPTIMIZATION RESULTS     
Total Iterations Run   : 50
Optimal Visiting Route : 0 -> 1 -> 3 -> 2 -> 0
Minimum Tour Distance  : 34.00 meters
